????? ?? ????? ???????. ??? ????? ?? ???? ??? ??? ???!!

In [ ]:
################## ?? ?? ?? ? ?? ##################
# ??? ??? ?? print() ?? ??
# ??) print(df.head())

# getcwd(), chdir() ? ?? ?? ?? ???
# ?? ?? ? ?? ???? ??(C: ?) ?? ??

# ??? ?? ?? ??
import pandas as pd
X_test = pd.read_csv("../input/hr-data/X_test.csv")
X_train = pd.read_csv("../input/hr-data/X_train.csv")
y_train = pd.read_csv("../input/hr-data/y_train.csv")

# ??? ??

# ?? ?? ??
# ?? ?? ????? ????? ???? ???? ??
# pd.DataFrame({'enrollee_id': X_test.enrollee_id, 'target': pred}).to_csv('003000000.csv', index=False)

# EDA

In [ ]:
# ??? ?? ??
X_train.shape, y_train.shape, X_test.shape

In [ ]:
# ??? ?? ??
X_train.head()

In [ ]:
# ?? ??? ?? ?? ?? (0? 1??? ????? ????? ????? ?? ??)
y_train.head()

In [ ]:
# ??? ??? ?? ??
y_train['target'].value_counts()

In [ ]:
# ??? ?? ??
print(X_train.info())

In [ ]:
# ??? ??
print("##Train##")
print(X_train.isnull().sum())
print("\n##Test##")
print(X_test.isnull().sum())

In [ ]:
# ???IQR
# training_hours
import numpy as np
Q1 = np.percentile(X_train['training_hours'],25)
Q3 = np.percentile(X_train['training_hours'],75)
IQR = Q3 - Q1
outdata1 = X_train[X_train['training_hours']<(Q1 - 1.5 * IQR)]
outdata2 = X_train[X_train['training_hours']>(Q3 + 1.5 * IQR)]
len(outdata1), len(outdata2)
# ???? ??

In [ ]:
# city_development_index       
Q1 = np.percentile(X_train['city_development_index'],25)
Q3 = np.percentile(X_train['city_development_index'],75)
IQR = Q3 - Q1
outdata1 = X_train[X_train['city_development_index']<(Q1 - 1.5 * IQR)]
outdata2 = X_train[X_train['city_development_index']>(Q3 + 1.5 * IQR)]
len(outdata1), len(outdata2)
#?? ??

In [ ]:
# ??? ??
obj_cols = np.array(X_train.columns[X_train.dtypes == object])
for col in obj_cols:
    print("\n##### : ",col)
    print(X_train[col].value_counts())
obj_cols

# ??? ???

In [ ]:
# ?? ??? ??
X_train = X_train.fillna("X")
X_test = X_test.fillna("X")

In [ ]:
# ??? ??
print(X_train.shape)
ind = X_train[X_train['city_development_index']<(Q1 - 1.5 * IQR)].index
X_train = X_train.drop(index=ind, axis=0)
y_train = y_train.drop(index=ind, axis=0)

print(X_train.shape)

# ?? ?????

In [ ]:
from sklearn.preprocessing import LabelEncoder

all_df = pd.concat([X_train.assign(ind="train"), X_test.assign(ind="test")])
le = LabelEncoder()
all_df[obj_cols] = all_df[obj_cols].apply(le.fit_transform)

X_train = all_df[all_df['ind'] == 'train']
X_train = X_train.drop('ind',axis=1)
X_train

X_test = all_df[all_df['ind'] == 'test']
X_test = X_test.drop('ind',axis=1)
X_test

In [ ]:
# ????
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()

n_cols = ['city_development_index', 'training_hours']
X_train[n_cols] = scaler.fit_transform(X_train[n_cols])
X_test[n_cols] = scaler.transform(X_test[n_cols])
X_train

# ??? ??

In [ ]:
# ??? ???? ??? ???? ??
# ??? ?? ???? ???? ?? ??? ?? ???
from sklearn.model_selection import train_test_split


X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train['target'], test_size=0.1, random_state=2022)
X_tr.shape, X_val.shape, y_tr.shape, y_val.shape

# ?? ?? ? ??

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
rf = RandomForestClassifier(random_state = 2022)
rf.fit(X_tr, y_tr)
pred = rf.predict_proba(X_val)[:,1]
print(roc_auc_score(y_val, pred))

# ???

In [ ]:
# Test???
pred = rf.predict_proba(X_test)[:,1]

In [ ]:
# csv????
pd.DataFrame({'enrollee_id': X_test.enrollee_id, 'target': pred}).to_csv('003000000.csv', index=False)

# ??

In [ ]:
# ??
import pickle
import numpy as np
from sklearn.metrics import roc_auc_score

with open( "../input/hr-data/answer.pickle", "rb" ) as file:
    ans = pickle.load(file)
    ans = pd.DataFrame(ans)
print(roc_auc_score(ans['target'], pred))